# PSW Sequence Manual Tests

PSW 四种时序功能手动联调。默认设备 `psw_01`，通道 `OUT`。

PSW 只有单通道 OUT，parallel/relative/staggered 在单台设备上主要验证接口链路。

顺序：
1. 运行初始化单元
2. 连通确认（IDN + measure）
3. 逐一测试四种 cycle，先预览再执行
4. 执行后检查设备面板和 runtime/sequence_logs/ 下的 CSV

In [ ]:
from __future__ import annotations
from pathlib import Path
import sys

def find_project_root(start):
    for c in [start, *start.parents]:
        if (c / 'pyproject.toml').exists() and (c / 'src' / 'power_control_host').exists():
            return c
    raise RuntimeError('未找到项目根目录')

PROJECT_ROOT = find_project_root(Path.cwd())
SRC_PATH = PROJECT_ROOT / 'src'
if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

CONFIG_PATH = PROJECT_ROOT / 'config' / 'devices.local.yaml'
DEVICE_ID = 'psw_01'  # 按实际配置修改
CHANNEL = 'OUT'
DEFAULT_LOG_DIR = PROJECT_ROOT / 'runtime' / 'sequence_logs'

print(f'PROJECT_ROOT = {PROJECT_ROOT}')
print(f'CONFIG_PATH  = {CONFIG_PATH}')
print(f'DEVICE_ID    = {DEVICE_ID}')
print(f'CHANNEL      = {CHANNEL}')

In [ ]:
from dataclasses import asdict
from datetime import datetime
try:
    import pandas as pd
except ImportError:
    pd = None
from IPython.display import display
from power_control_host.app import create_app
from power_control_host.models import ChannelCycleSpec, RelativeChannelSpec

app = create_app(CONFIG_PATH)
print('configured_devices:', app.device_service.list_devices())
print('logical_channels:', app.device_service.get_logical_channels(DEVICE_ID))

def refresh_app():
    global app
    app = create_app(CONFIG_PATH)
    print('app reloaded')

def show_plan(plan):
    rows = [{'step': i, 'channel': s.channel, 'action': s.action,
             'delay_s': s.delay_seconds, 'voltage': s.voltage, 'current': s.current}
            for i, s in enumerate(plan.steps, 1)]
    print(f'plan: {plan.name}  steps: {len(rows)}')
    if pd is not None:
        display(pd.DataFrame(rows))
    else:
        for r in rows: print(r)
    return rows

def make_log_path(label):
    ts = datetime.now().strftime('%Y%m%d_%H%M%S')
    return DEFAULT_LOG_DIR / f'{label}_{ts}.csv'

def run_plan(plan, label=None):
    log_path = make_log_path(label or plan.name)
    events = app.sequence_service.execute_plan(plan, log_path=log_path)
    print(f'executed: {plan.name}  events: {len(events)}  log: {log_path}')
    rows = [asdict(e) for e in events]
    if pd is not None:
        display(pd.DataFrame(rows))
    else:
        for r in rows: print(r)
    return events, log_path

def measure():
    sample = app.device_service.read_measurement(DEVICE_ID, CHANNEL)
    for k, v in asdict(sample).items(): print(f'{k}: {v}')
    return sample

## 0. 连通确认

In [ ]:
identity = app.device_service.identify(DEVICE_ID)
print(f'device_id: {identity.device_id}')
print(f'model:     {identity.model}')
print(f'identity:  {identity.identity}')

In [ ]:
measure()

## 1. 单通道循环

对应 CLI: `run-cycle --device psw_01 --channel OUT`

In [ ]:
single_on_seconds  = 5.0
single_off_seconds = 5.0
single_cycles      = 3
single_voltage     = 12.0
single_current     = 2.0

In [ ]:
single_plan = app.sequence_service.build_single_channel_cycle_plan(
    device_id=DEVICE_ID, channel=CHANNEL,
    on_seconds=single_on_seconds, off_seconds=single_off_seconds,
    cycles=single_cycles, voltage=single_voltage, current=single_current,
)
show_plan(single_plan)

In [ ]:
run_plan(single_plan, 'psw_single_cycle')

In [ ]:
measure()

## 2. 同起点独立循环

对应 CLI: `run-parallel-cycle`

PSW 单通道，传一个 spec，主要验证接口链路。

In [ ]:
parallel_specs = [
    ChannelCycleSpec(channel=CHANNEL, on_seconds=5.0, off_seconds=5.0, cycles=2, voltage=12.0, current=2.0),
]

In [ ]:
parallel_plan = app.sequence_service.build_parallel_channel_cycle_plan(
    device_id=DEVICE_ID, channel_specs=parallel_specs)
show_plan(parallel_plan)

In [ ]:
run_plan(parallel_plan, 'psw_parallel_cycle')

## 3. 相对时序循环

对应 CLI: `run-relative-cycle`

单通道无 ref 延迟，等价于单通道循环，主要验证接口链路。

In [ ]:
relative_on_seconds  = 8.0
relative_off_seconds = 4.0
relative_cycles      = 2
relative_specs = [
    RelativeChannelSpec(channel=CHANNEL, voltage=12.0, current=2.0),
]

In [ ]:
relative_plan = app.sequence_service.build_relative_channel_cycle_plan(
    device_id=DEVICE_ID, on_seconds=relative_on_seconds,
    off_seconds=relative_off_seconds, cycles=relative_cycles,
    channel_specs=relative_specs,
)
show_plan(relative_plan)

In [ ]:
run_plan(relative_plan, 'psw_relative_cycle')

## 4. 后上先下兼容入口（staggered）

对应 CLI: `run-staggered-cycle`

单通道 lead/lag 均传 OUT，delay=0，等价于单通道循环，主要验证接口链路。

In [ ]:
staggered_delay   = 0.0
staggered_hold    = 5.0
staggered_rest    = 3.0
staggered_cycles  = 2
staggered_voltage = 12.0
staggered_current = 2.0

In [ ]:
staggered_plan = app.sequence_service.build_staggered_channel_cycle_plan(
    device_id=DEVICE_ID,
    lead_channel=CHANNEL, lag_channel=CHANNEL,
    delay_seconds=staggered_delay, hold_seconds=staggered_hold,
    rest_seconds=staggered_rest, cycles=staggered_cycles,
    lead_voltage=staggered_voltage, lead_current=staggered_current,
    lag_voltage=staggered_voltage, lag_current=staggered_current,
)
show_plan(staggered_plan)

In [ ]:
run_plan(staggered_plan, 'psw_staggered_cycle')

## 说明

- 改了 devices.local.yaml 后执行 refresh_app() 重载
- 日志写到 runtime/sequence_logs/，检查 CSV 里动作顺序和时间间隔
- parallel/relative/staggered 在 PSW 单通道下主要验证接口链路